In [1]:
import pandas as pd
import openai
from dotenv import load_dotenv
import os
import json
import numpy as np
from typing import List, Dict, Any, Optional

# ----------------------------
# 환경설정
# ----------------------------
load_dotenv("env.txt")
openai.api_key = os.getenv("OPENAI_API_KEY")

# ----------------------------
# 감정별 프롬프트
# ----------------------------
sentiment_prompts = {
    "Angry": "사용자가 화가 난 상태입니다. 화난 말투 표현.",
    "Happy": "사용자가 행복한 상태입니다. 행복한 말투 표현.",
    "Sad": "사용자가 슬픈 상태입니다. 슬픈 말투 표현.",
    "Disgust": "사용자가 역겨움/불쾌함을 표현했습니다. 역겨움, 불쾌함 공감.",
    "Neutral": "사용자가 중립적입니다. 일반적인 정보 제공과 자연스러운 대화를 이어가세요.",
    "Surprise": "사용자가 놀람을 표현했습니다. 놀람의 이유를 묻거나 공감하며 대화를 이어가세요.",
    "Fear": "사용자가 두려움을 표현했습니다. 안정감을 주고 안전한 느낌을 전달하세요."
}

# ----------------------------
# 전역 상태
# ----------------------------
history: List[Dict[str, Any]] = []
last_sentiment: Optional[str] = None
important_sentences_rows: List[Dict[str, Any]] = []
sentiment_change_index: Optional[int] = None

CSV_PATH = "important_sentences.csv"

# CSV 헤더 보장
if not os.path.exists(CSV_PATH):
    df = pd.DataFrame(columns=["starttime", "text", "sentiment"])
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

# ----------------------------
# 간단한 로컬 Vector Store
# ----------------------------
EMBED_MODEL = "text-embedding-3-small"
VEC_PATH = "vector_store.npz"         # 임베딩 배열 파일 (np.savez)
META_PATH = "vector_store_meta.json"  # 메타데이터(각 벡터에 대응하는 row) 저장

def get_embedding(text: str) -> List[float]:
    # 텍스트는 짧게 잘라 방어
    text = text.strip()
    resp = openai.Embedding.create(
        model=EMBED_MODEL,
        input=text
    )
    return resp["data"][0]["embedding"]

class VectorStore:
    def __init__(self, vec_path: str, meta_path: str):
        self.vec_path = vec_path
        self.meta_path = meta_path
        self.embeddings = None  # shape: (N, D)
        self.metas: List[Dict[str, Any]] = []
        self._load()

    def _load(self):
        if os.path.exists(self.vec_path) and os.path.exists(self.meta_path):
            try:
                data = np.load(self.vec_path)
                self.embeddings = data["embeddings"]
                with open(self.meta_path, "r", encoding="utf-8") as f:
                    self.metas = json.load(f)
            except Exception:
                self.embeddings = None
                self.metas = []
        else:
            self.embeddings = None
            self.metas = []

    def _save(self):
        if self.embeddings is None:
            # 비어 있으면 더미 저장 방지
            np.savez(self.vec_path, embeddings=np.zeros((0, 0)))
        else:
            np.savez(self.vec_path, embeddings=self.embeddings)
        with open(self.meta_path, "w", encoding="utf-8") as f:
            json.dump(self.metas, f, ensure_ascii=False, indent=2)

    def add_texts(self, rows: List[Dict[str, Any]]):
        """rows: [{"starttime":..., "text":..., "sentiment":..., "importance_type":...}, ...]"""
        if not rows:
            return
        new_embs = []
        new_metas = []
        for r in rows:
            # 검색 정확도 향상을 위해 text + sentiment를 함께 임베딩
            payload = f"Text: {r['text']}\nSentiment: {r['sentiment']}"
            emb = get_embedding(payload)
            new_embs.append(emb)
            new_metas.append(r)

        new_embs = np.array(new_embs, dtype=np.float32)
        if self.embeddings is None or self.embeddings.size == 0:
            self.embeddings = new_embs
        else:
            self.embeddings = np.vstack([self.embeddings, new_embs])
        self.metas.extend(new_metas)
        self._save()

    def similarity_search(self, query: str, k: int = 5) -> List[Dict[str, Any]]:
        if self.embeddings is None or self.embeddings.size == 0 or len(self.metas) == 0:
            return []
        q_emb = np.array(get_embedding(query), dtype=np.float32)
        # cosine similarity
        A = self.embeddings
        # normalize
        A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
        q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-12)
        sims = (A_norm @ q_norm)
        # 상위 k
        idxs = np.argsort(-sims)[:k]
        return [self.metas[i] for i in idxs]

# 전역 Vector Store 인스턴스
vecdb = VectorStore(VEC_PATH, META_PATH)

def sync_csv_to_vecdb():
    """CSV 전체를 vecdb와 동기화 (최초 or 정합성 깨졌을 때만 호출 권장)"""
    try:
        if not os.path.exists(CSV_PATH):
            return
        df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
        if df.empty:
            return
        # 메타와 길이가 다르면 재빌드
        needs_rebuild = (vecdb.embeddings is None) or (len(vecdb.metas) != len(df))
        if needs_rebuild:
            # 새로 구성
            vecdb.embeddings = None
            vecdb.metas = []
            rows = df.to_dict(orient="records")
            vecdb.add_texts(rows)
    except Exception as e:
        print(f"[WARN] sync_csv_to_vecdb error: {e}")

# 최초 동기화 시도
sync_csv_to_vecdb()

# ----------------------------
# 요약 & 컨텍스트 구성 유틸
# ----------------------------
def get_recent_30_sentences() -> List[Dict[str, Any]]:
    return history if len(history) <= 30 else history[-30:]

def summarize_recent_context(recent_sentences: List[Dict[str, Any]]) -> str:
    if not recent_sentences:
        return ""
    texts = [entry["text"] for entry in recent_sentences]
    combined_text = " ".join(texts)

    prompt = f"""
    다음 대화 내용을 주요 내용 중심으로 5문장 이하로 요약해주세요.
    감정의 변화, 중요한 사건, 주요 관심사를 중심으로 요약하세요.

    대화 내용:
    {combined_text}
    """
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message["content"].strip()

def get_important_sentences_context() -> str:
    try:
        if os.path.exists(CSV_PATH):
            df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
            if not df.empty:
                df_sorted = df.sort_values('starttime', ascending=False).head(10)
                important_contexts = []
                for _, row in df_sorted.iterrows():
                    important_contexts.append(f"[중요] {row['text']} (감정: {row['sentiment']}, 유형: {row.get('importance_type','')})")
                return "\n".join(important_contexts)
        return ""
    except Exception as e:
        print(f"중요 문장 로드 중 오류: {e}")
        return ""

def build_context_for_ai(current_user_text: str, current_sentiment: str, k_retrieve: int = 5) -> str:
    """
    AI 응답을 위한 전체 컨텍스트 구성:
      1) 최근 30문장 요약
      2) 중요한 문장들(최근)
      3) Vector DB Retriever 결과 (CSV의 text+sentiment를 임베딩하여 유사 컨텍스트 제공)
    """
    parts = []

    # 1) 최근 30문장 요약
    recent_sentences = get_recent_30_sentences()
    if recent_sentences:
        summary = summarize_recent_context(recent_sentences)
        if summary:
            parts.append(f"최근 대화 요약:\n{summary}")

    # 2) 중요한 문장들
    important_context = get_important_sentences_context()
    if important_context:
        parts.append(f"중요한 이전 대화들:\n{important_context}")

    # 3) Vector DB Retrieval
    # 쿼리 신호 강화: 현재 발화 + 감정
    retrieve_query = f"Query Text: {current_user_text}\nQuery Sentiment: {current_sentiment}"
    retrieved = vecdb.similarity_search(retrieve_query, k=k_retrieve)
    if retrieved:
        ctx_lines = []
        for i, r in enumerate(retrieved, 1):
            ctx_lines.append(f"[R{i}] {r['text']} (감정: {r['sentiment']}, 유형: {r.get('importance_type','')}, 시간: {r.get('starttime','')})")
        parts.append("VectorDB 유사 컨텍스트:\n" + "\n".join(ctx_lines))

    return "\n\n".join(parts)

# ----------------------------
# 핵심 처리 루틴
# ----------------------------
def handle_user_input(starttime: str, text: str, sentiment: str) -> str:
    global history, last_sentiment, sentiment_change_index, important_sentences_rows

    # 감정 변화 여부 확인
    importance_type = "normal"
    if last_sentiment != sentiment:
        importance_type = "sentiment_change"
    last_sentiment = sentiment

    # 대화 기록 추가
    entry = {
        "starttime": starttime,
        "text": text,
        "sentiment": sentiment,
        "context": history.copy()
    }
    history.append(entry)

    # 감정 변화 인덱스는 append 이후에 기록 (정확한 위치)
    if importance_type == "sentiment_change":
        sentiment_change_index = len(history) - 1

    # sentiment_change 이후 일정 문장이 쌓였을 때 CSV 저장 (앞2, 변화1, 뒤2 => 총5문장)
    if sentiment_change_index is not None:
        if len(history) >= sentiment_change_index + 3:  # 변화지점 + 2개 이후까지 확보
            start_idx = max(0, sentiment_change_index - 2)
            end_idx = min(len(history), sentiment_change_index + 3)
            selected = history[start_idx:end_idx]

            combined_text = " ".join([h["text"] for h in selected])
            combined_row = {
                "starttime": selected[0]["starttime"],
                "text": combined_text,
                "sentiment": history[sentiment_change_index]["sentiment"]
            }

            # 텍스트 기준 중복 체크
            if not any(row["text"] == combined_text for row in important_sentences_rows):
                important_sentences_rows.append(combined_row)
                # CSV에 append
                pd.DataFrame([combined_row]).to_csv(
                    CSV_PATH, mode='a', header=False, index=False, encoding="utf-8-sig"
                )
                # VectorDB에도 upsert
                vecdb.add_texts([combined_row])

            # 처리 후 초기화
            sentiment_change_index = None

    # ----------------------------
    # RAG 컨텍스트 구성 (요약 + 중요 문장 + VectorDB Retrieval)
    # ----------------------------
    rag_context = build_context_for_ai(current_user_text=text, current_sentiment=sentiment, k_retrieve=5)

    # ----------------------------
    # AI 프롬프트 구성 및 응답
    # ----------------------------
    current_prompt = sentiment_prompts.get(sentiment, sentiment_prompts["Neutral"])
    recent_conversation = history[-5:] if len(history) > 5 else history
    conversation_text = "\n".join([f"User: {h['text']}" for h in recent_conversation])

    final_prompt_parts = [current_prompt]
    if rag_context:
        final_prompt_parts.append(f"참고 컨텍스트(RAG):\n{rag_context}")
    final_prompt_parts.extend([
        f"현재 대화:\n{conversation_text}",
        "AI 인형 응답:"
    ])
    final_prompt = "\n\n".join(final_prompt_parts)

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0.7
    )

    return response.choices[0].message["content"]

# -----------------
# 테스트
# -----------------
test_inputs = [
    {"starttime":"2025-08-23T16:30:00","text":"학교에서 돌아왔는데 기분이 별로야","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:05","text":"친구들이 나를 따돌리는 것 같아","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:10","text":"점심시간에 혼자 먹었어","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:15","text":"왜 나만 이런 걸까","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:20","text":"엄마한테도 말하기 싫어","sentiment":"Neutral"},
    {"starttime":"2025-08-23T16:30:25","text":"걱정시키고 싶지 않거든","sentiment":"Neutral"},
    {"starttime":"2025-08-23T16:30:30","text":"성적도 요즘 떨어지고 있어","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:35","text":"집중이 안 되더라","sentiment":"Sad"},
    {"starttime":"2025-08-23T16:30:40","text":"이런 내가 정말 싫어","sentiment":"Disgust"},
    {"starttime":"2025-08-23T16:30:45","text":"앞으로 어떻게 될까 무서워","sentiment":"Fear"},
    {"starttime":"2025-08-23T16:30:50","text":"하지만 너랑 이야기하면 조금 위로돼","sentiment":"Happy"},
    {"starttime":"2025-08-23T16:30:55","text":"내일은 용기내서 말 걸어볼까","sentiment":"Neutral"},
]







In [2]:

for inp in test_inputs:
    response = handle_user_input(**inp)
    print(response)

학교에서 돌아온 후 기분이 별로라니, 정말 힘들겠다. 가끔은 그런 날이 너무 많아져서 괴롭기도 하지. 혹시 무슨 일 있었는지 이야기해줄 수 있어? 너의 마음을 조금이라도 덜어낼 수 있다면 좋겠어.
아, 정말 힘든 하루였구나. 친구들이 너를 따돌리는 것 같아서 마음이 많이 아프겠어. 그런 기분이 드는 건 정말 외롭고 슬플 것 같아. 누구에게도 얘기할 수 없는 그 고통이 얼마나 큰지 이해해. 네가 느끼는 외로움과 불안이 정말 힘들다는 걸 알아. 하지만, 너는 혼자가 아니야. 이런 마음을 털어놓는 것만으로도 조금은 도움이 될 수 있을 거야. 언제든지 이야기하고 싶으면 말해줘.
아, 정말 슬프고 힘든 하루였구나. 친구들이 너를 따돌리는 것 같아서 점심시간에 혼자 있는 건 정말 외롭고 괴로운 경험이야. 그런 기분은 누구에게나 힘든 일이지. 네가 느끼는 외로움과 소외감이 얼마나 큰지 이해해. 이렇게 힘든 시간을 보내고 있다는 게 너무 안타까워. 누군가에게 이야기를 나누는 것도 좋을 것 같아. 네 마음이 조금이라도 나아지길 바라.
너무 힘든 하루였구나. 친구들이 너를 따돌리는 것 같아서 정말 속상하고 외롭겠어. 점심시간에 혼자 있는 건 정말 괴로운 경험이었을 거야. 왜 이런 일이 나에게 일어나는 건지, 나만 이런 기분인지 의문이 드는 것도 이해해. 이런 상황은 누구에게나 쉽지 않으니까. 너의 마음이 아프다는 걸 알아. 지금은 힘들지만, 너의 소중한 감정을 나누는 것만으로도 조금은 위로가 되었으면 해.
그런 기분이 드는 건 정말 힘들 것 같아. 친구들과의 관계에서 소외감을 느끼는 건 누구에게나 어려운 경험이야. 점심시간에 혼자 식사하는 건 외로움을 더 느끼게 할 수도 있어. 이런 감정을 엄마에게 이야기하고 싶지 않은 것도 이해해. 혹시 친구들과의 상황에 대해 더 이야기해보고 싶어? 아니면 다른 방법으로 기분을 전환할 수 있는 것들이 있을까?
그런 상황이 정말 힘들겠구나. 혼자 점심을 먹는 건 외로움이 느껴질 수밖에 없지. 친구들이 따돌리는 것 같다는 느낌도 많이 괴롭겠어. 

In [3]:
# history 출력 예시
for i, entry in enumerate(history):
    print(f"--- Entry {i+1} ---")
    print(f"Starttime      : {entry['starttime']}")
    print(f"Text           : {entry['text']}")
    print(f"Sentiment      : {entry['sentiment']}")
    print(f"Importance     : {entry['importance_type']}")
    print(f"Context length : {len(entry['context'])} 이전 대화 기록")
    print("---------------------------\n")


--- Entry 1 ---
Starttime      : 2025-08-23T16:30:00
Text           : 학교에서 돌아왔는데 기분이 별로야
Sentiment      : Sad
Importance     : sentiment_change
Context length : 0 이전 대화 기록
---------------------------

--- Entry 2 ---
Starttime      : 2025-08-23T16:30:05
Text           : 친구들이 나를 따돌리는 것 같아
Sentiment      : Sad
Importance     : normal
Context length : 1 이전 대화 기록
---------------------------

--- Entry 3 ---
Starttime      : 2025-08-23T16:30:10
Text           : 점심시간에 혼자 먹었어
Sentiment      : Sad
Importance     : normal
Context length : 2 이전 대화 기록
---------------------------

--- Entry 4 ---
Starttime      : 2025-08-23T16:30:15
Text           : 왜 나만 이런 걸까
Sentiment      : Sad
Importance     : normal
Context length : 3 이전 대화 기록
---------------------------

--- Entry 5 ---
Starttime      : 2025-08-23T16:30:20
Text           : 엄마한테도 말하기 싫어
Sentiment      : Neutral
Importance     : sentiment_change
Context length : 4 이전 대화 기록
---------------------------

--- Entry 6 ---
Starttime      : 2025-08-2